# 04 — Phase 3: League Analysis
**Football Players Stats 2024/25**

Comparing the 5 major European leagues across key metrics.

> Uses `league_df` filtered to 900+ minutes for fair comparison.

## Setup

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

league_df = pd.read_csv('Dataset\league_clean.csv')

LEAGUE_COLORS = {
    'Premier League': '#3d195b', 'La Liga': '#ee8707',
    'Serie A': '#008fd7', 'Bundesliga': '#d3010c', 'Ligue 1': '#091c3e',
}

league_ranked = league_df[league_df['minutes_played'] >= 900].copy()
print(f"Players with 900+ minutes: {len(league_ranked)}")

Players with 900+ minutes: 1570


In [2]:
# 3.1 Total Goals by League
total_goals = league_ranked.groupby('league')['goals'].sum().reset_index()
total_goals = total_goals.sort_values('goals', ascending=False)

fig = px.bar(total_goals, x='league', y='goals', color='league',
             color_discrete_map=LEAGUE_COLORS,
             title='Total Goals Scored per League (2024/25)',
             labels={'goals': 'Total Goals', 'league': ''}, text='goals')
fig.update_traces(textposition='outside')
fig.update_layout(height=450, showlegend=False, plot_bgcolor='white',
                  yaxis=dict(showgrid=True, gridcolor='#f0f0f0'))
fig.show()

In [3]:
# 3.2 Average Goals per Player
avg_goals = league_ranked.groupby('league')['goals'].mean().reset_index()
avg_goals.columns = ['league', 'avg_goals']
avg_goals['avg_goals'] = avg_goals['avg_goals'].round(2)
avg_goals = avg_goals.sort_values('avg_goals', ascending=False)

fig = px.bar(avg_goals, x='league', y='avg_goals', color='league',
             color_discrete_map=LEAGUE_COLORS,
             title='Average Goals per Player per League (2024/25)',
             labels={'avg_goals': 'Average Goals', 'league': ''}, text='avg_goals')
fig.update_traces(textposition='outside')
fig.update_layout(height=450, showlegend=False, plot_bgcolor='white',
                  yaxis=dict(showgrid=True, gridcolor='#f0f0f0'))
fig.show()

In [4]:
# 3.3 Average Pass Completion %
avg_pass = league_ranked.groupby('league')['pass_completion_pct'].mean().reset_index()
avg_pass.columns = ['league', 'avg_pass_pct']
avg_pass['avg_pass_pct'] = avg_pass['avg_pass_pct'].round(1)
avg_pass = avg_pass.sort_values('avg_pass_pct', ascending=False)

fig = px.bar(avg_pass, x='league', y='avg_pass_pct', color='league',
             color_discrete_map=LEAGUE_COLORS,
             title='Average Pass Completion % per League (2024/25)',
             labels={'avg_pass_pct': 'Pass Completion %', 'league': ''}, text='avg_pass_pct')
fig.update_traces(textposition='outside', texttemplate='%{text}%')
fig.update_layout(height=450, showlegend=False, plot_bgcolor='white',
                  yaxis=dict(showgrid=True, gridcolor='#f0f0f0', range=[70, 90]))
fig.show()

In [5]:
# 3.4 Average Tackles + Interceptions (Defensive Intensity)
avg_def = league_ranked.groupby('league')[['tackles', 'interceptions']].mean().reset_index().round(1)
avg_def = avg_def.sort_values('tackles', ascending=False)

fig = go.Figure()
fig.add_trace(go.Bar(name='Avg Tackles', x=avg_def['league'], y=avg_def['tackles'],
                     marker_color=[LEAGUE_COLORS[l] for l in avg_def['league']],
                     text=avg_def['tackles'], textposition='outside'))
fig.add_trace(go.Bar(name='Avg Interceptions', x=avg_def['league'], y=avg_def['interceptions'],
                     marker_color=[LEAGUE_COLORS[l] for l in avg_def['league']],
                     opacity=0.5, text=avg_def['interceptions'], textposition='outside'))
fig.update_layout(title='Average Tackles & Interceptions per Player per League (2024/25)',
                  barmode='group', height=450, plot_bgcolor='white',
                  yaxis=dict(showgrid=True, gridcolor='#f0f0f0'),
                  yaxis_title='Average per Player', legend_title='Metric')
fig.show()

In [6]:
# 3.5 Average Age by League
avg_age = league_ranked.groupby('league')['age'].mean().reset_index()
avg_age.columns = ['league', 'avg_age']
avg_age['avg_age'] = avg_age['avg_age'].round(1)
avg_age = avg_age.sort_values('avg_age', ascending=True)

fig = px.bar(avg_age, x='avg_age', y='league', orientation='h', color='league',
             color_discrete_map=LEAGUE_COLORS,
             title='Average Player Age per League (2024/25)',
             labels={'avg_age': 'Average Age', 'league': ''}, text='avg_age')
fig.update_traces(textposition='outside')
fig.update_layout(height=400, showlegend=False, plot_bgcolor='white',
                  xaxis=dict(showgrid=True, gridcolor='#f0f0f0', range=[24, 30]))
fig.show()

In [7]:
# 3.6 Normalized Multi-Metric League Comparison
metrics = {
    'avg_goals':    league_ranked.groupby('league')['goals'].mean(),
    'avg_assists':  league_ranked.groupby('league')['assists'].mean(),
    'avg_shots':    league_ranked.groupby('league')['shots'].mean(),
    'avg_pass_pct': league_ranked.groupby('league')['pass_completion_pct'].mean(),
    'avg_tackles':  league_ranked.groupby('league')['tackles'].mean(),
}

summary = pd.DataFrame(metrics).reset_index()
summary.columns = ['league', 'Goals', 'Assists', 'Shots', 'Pass Completion %', 'Tackles']
for col in ['Goals', 'Assists', 'Shots', 'Pass Completion %', 'Tackles']:
    summary[col] = ((summary[col] - summary[col].min()) /
                    (summary[col].max() - summary[col].min()) * 100).round(1)

summary_melted = summary.melt(id_vars='league', var_name='metric', value_name='score')
fig = px.bar(summary_melted, x='metric', y='score', color='league', barmode='group',
             color_discrete_map=LEAGUE_COLORS,
             title='League Comparison — Normalized Score per Metric (2024/25)',
             labels={'score': 'Normalized Score (0-100)', 'metric': '', 'league': 'League'})
fig.update_layout(height=500, plot_bgcolor='white',
                  yaxis=dict(showgrid=True, gridcolor='#f0f0f0'), legend_title='League')
fig.show()